In [1]:
# Install lerobot and huggingface - note this will install the latest
!pip install -U lerobot[dataset] huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 16.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 71.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 74.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 79.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 45.9 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: packaging
    Found existing installation: packaging 26.2
    Uninstalling packaging-26.2:
      Successfully uninstalled packaging-26.2
  Attem

In [2]:
from huggingface_hub import login
login(token="YOUR_HF_TOKEN")

# Clear cache
import shutil
from pathlib import Path
cache_dir = Path("/root/.cache/huggingface/lerobot")
if cache_dir.exists():
    shutil.rmtree(cache_dir)
    print("Cleared old LeRobot cache!")

In [19]:
# Using the native PyTorch API gives you full control over the training loop!
import torch
import copy
from torch.utils.data import DataLoader
from lerobot.configs.types import FeatureType
from lerobot.utils.feature_utils import dataset_to_policy_features
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.policies.diffusion.configuration_diffusion import DiffusionConfig
from lerobot.policies.diffusion.modeling_diffusion import DiffusionPolicy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#TODO: Fill the below with your own credential
HF_REPO = "hf_user_id/repo_id"
HF_TRAIN_REPO = "hf_user_id/trained_repo_id" # where your trained policy will be stored
HORIZON = 16 
IMAGE_HZ = 10

# 1. Define horizons for Diffusion Policy (Assuming 10Hz data collection)
delta_timestamps = {
    # Give the model context: The frame from 0.1s ago, and the current frame
    "observation.image": [-1.0/IMAGE_HZ, 0.0], 
    # Predict the next 16 steps (1.6 seconds into the future)
    "action": [t * 1.0/IMAGE_HZ for t in range(HORIZON)] 
}

# 2. Load the dataset with the temporal window applied
dataset = LeRobotDataset(
    HF_REPO,
    revision="main",           
    delta_timestamps=delta_timestamps
)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=2)

# 3. Dynamically extract the features from the dataset
input_features, output_features = dataset_to_policy_features(dataset.meta.features)

features = dataset_to_policy_features(dataset.meta.features)
# Isolate the actions for the output
output_features = {k: ft for k, ft in features.items() if ft.type is FeatureType.ACTION}
# Everything else goes to the input
input_features = {k: ft for k, ft in features.items() if k not in output_features}

# LeRobot's Diffusion Policy assumes a robot state (joint angles) is ALWAYS present.
# We inject a dummy state config here so it doesn't crash on pure-visual datasets.
dummy_state = copy.deepcopy(output_features["action"])
dummy_state.type = FeatureType.STATE
dummy_state.shape = (1,) # A single dummy value dimension
input_features["observation.state"] = dummy_state

dataset.meta.stats["observation.state"] = {
    "min": torch.tensor([0.0]), "max": torch.tensor([1.0]),
    "mean": torch.tensor([0.0]), "std": torch.tensor([1.0])
}

# 4. Initialize the Diffusion Policy
action_len = len(delta_timestamps["action"])
obs_len = len(delta_timestamps["observation.image"])
cfg = DiffusionConfig(
    input_features=input_features, 
    output_features=output_features,
    horizon=action_len,  # MUST match the length of our action delta_timestamps
    n_obs_steps=obs_len, # MUST match the length of our image delta_timestamps
    n_action_steps=8     # How many steps to actually execute in inference before re-planning
)
policy = DiffusionPolicy(cfg, dataset_stats=dataset.meta.stats)
policy.to(device)
policy.train()

# 5. Setup Optimizer (Diffusion models often use slightly higher LRs like 1e-4)
optimizer = torch.optim.AdamW(policy.parameters(), lr=1e-4)

print("Starting Diffusion Policy training...")
for epoch in range(50):
    for batch in dataloader:
        # Move all tensors to the GPU
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items() if isinstance(v, torch.Tensor)}
        
        # Inject the dummy state and padding mask into the live batch
        batch_size = batch["action"].shape[0]
        
        # DP expects the state to match the image horizon (2 steps: -0.1s and 0.0s)
        batch["observation.state"] = torch.zeros((batch_size, obs_len, 1), dtype=torch.float32, device=device)
        
        # DP also strictly expects an "action_is_pad" boolean mask. 
        # We predict 16 steps into the future, so we provide a 16-step valid mask.
        batch["action_is_pad"] = torch.zeros((batch_size, action_len), dtype=torch.bool, device=device)
        
        # Forward pass: LeRobot automatically calculates the denoising loss for the batch
        loss, _ = policy(batch)
        
        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

# 6. Save your trained model to Hugging Face
policy.push_to_hub(HF_TRAINED_REPO)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Starting Diffusion Policy training...
Epoch 0 | Loss: 0.0070
Epoch 1 | Loss: 0.0019
Epoch 2 | Loss: 0.0020
Epoch 3 | Loss: 0.0017
Epoch 4 | Loss: 0.0027
Epoch 5 | Loss: 0.0017
Epoch 6 | Loss: 0.0010
Epoch 7 | Loss: 0.0033
Epoch 8 | Loss: 0.0010
Epoch 9 | Loss: 0.0014
Epoch 10 | Loss: 0.0011
Epoch 11 | Loss: 0.0007
Epoch 12 | Loss: 0.0013
Epoch 13 | Loss: 0.0007
Epoch 14 | Loss: 0.0006
Epoch 15 | Loss: 0.0013
Epoch 16 | Loss: 0.0020
Epoch 17 | Loss: 0.0007
Epoch 18 | Loss: 0.0002
Epoch 19 | Loss: 0.0006
Epoch 20 | Loss: 0.0007
Epoch 21 | Loss: 0.0012
Epoch 22 | Loss: 0.0002
Epoch 23 | Loss: 0.0002
Epoch 24 | Loss: 0.0003
Epoch 25 | Loss: 0.0002
Epoch 26 | Loss: 0.0005
Epoch 27 | Loss: 0.0008
Epoch 28 | Loss: 0.0001
Epoch 29 | Loss: 0.0003
Epoch 30 | Loss: 0.0003
Epoch 31 | Loss: 0.0006
Epoch 32 | Loss: 0.0005
Epoch 33 | Loss: 0.0004
Epoch 34 | Loss: 0.0004
Epoch 35 | Loss: 0.0005
Epoch 36 | Loss: 0.0001
Epoch 37 | Loss: 0.0015
Epoch 38 | Loss: 0.0001
Epoch 39 | Loss: 0.0002
Epoch 40 | L

CommitInfo(commit_url='https://huggingface.co/yoonheeh/trained_homebot_diffusion_policy/commit/653f6fba5bb40fff7ed2365bef2d3acbfb64d102', commit_message='Upload policy', commit_description='', oid='653f6fba5bb40fff7ed2365bef2d3acbfb64d102', pr_url=None, repo_url=RepoUrl('https://huggingface.co/yoonheeh/trained_homebot_diffusion_policy', endpoint='https://huggingface.co', repo_type='model', repo_id='yoonheeh/trained_homebot_diffusion_policy'), pr_revision=None, pr_num=None)

In [ ]:
# This version is to leverage multiple available GPUs (e.g. Kaggle's T4 x2 GPUs)
import torch
import copy
from torch.utils.data import DataLoader
from lerobot.configs.types import FeatureType
from lerobot.utils.feature_utils import dataset_to_policy_features
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.policies.diffusion.configuration_diffusion import DiffusionConfig
from lerobot.policies.diffusion.modeling_diffusion import DiffusionPolicy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#TODO: Fill the below with your own credential
HF_REPO = "hf_user_id/repo_id"
HF_TRAIN_REPO = "hf_user_id/trained_repo_id" # where your trained policy will be stored

# 1. Define horizons for Diffusion Policy (Assuming 10Hz data collection)
delta_timestamps = {
    # Give the model context: The frame from 0.1s ago, and the current frame
    "observation.image": [-0.1, 0.0], 
    # Predict the next 16 steps (1.6 seconds into the future)
    "action": [t * 0.1 for t in range(16)] 
}

# 2. Load the dataset with the temporal window applied
dataset = LeRobotDataset(
    "yoonheeh/homebot_15",
    revision="main",           
    delta_timestamps=delta_timestamps
)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=2)

# 3. Dynamically extract the features from the dataset
features = dataset_to_policy_features(dataset.meta.features)
output_features = {k: ft for k, ft in features.items() if ft.type is FeatureType.ACTION}
input_features = {k: ft for k, ft in features.items() if k not in output_features}

dummy_state = copy.deepcopy(output_features["action"])
dummy_state.type = FeatureType.STATE
dummy_state.shape = (1,) 
input_features["observation.state"] = dummy_state

dataset.meta.stats["observation.state"] = {
    "min": torch.tensor([0.0]), "max": torch.tensor([1.0]),
    "mean": torch.tensor([0.0]), "std": torch.tensor([1.0])
}

# 4. Initialize the Diffusion Policy
cfg = DiffusionConfig(
    input_features=input_features, 
    output_features=output_features,
    horizon=16,        
    n_obs_steps=2,     
    n_action_steps=8   
)
policy = DiffusionPolicy(cfg, dataset_stats=dataset.meta.stats)

# MULTI-GPU WRAPPER
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    policy = torch.nn.DataParallel(policy)

policy.to(device)
policy.train()

# 5. Setup Optimizer (Diffusion models often use slightly higher LRs like 1e-4)
optimizer = torch.optim.AdamW(policy.parameters(), lr=1e-4)

print("Starting Diffusion Policy training...")
for epoch in range(50):
    for batch in dataloader:
        # Move all tensors to the primary GPU (DataParallel will handle scattering them)
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items() if isinstance(v, torch.Tensor)}
        
        # Inject the dummy state and padding mask into the live batch
        batch_size = batch["action"].shape[0]
        batch["observation.state"] = torch.zeros((batch_size, 2, 1), dtype=torch.float32, device=device)
        batch["action_is_pad"] = torch.zeros((batch_size, 16), dtype=torch.bool, device=device)
        
        # Forward pass
        loss, _ = policy(batch)
        
        # MULTI-GPU LOSS AVERAGING 
        # DataParallel returns an array of losses (one from each GPU). 
        # We must average them into a single scalar before calling .backward()
        if torch.cuda.device_count() > 1:
            loss = loss.mean()
        
        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

# 6. Save your trained model to Hugging Face
# We have to extract the underlying model out of the DataParallel wrapper to save it properly
model_to_save = policy.module if torch.cuda.device_count() > 1 else policy
model_to_save.push_to_hub(HF_TRAINED_REPO)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Using 2 GPUs!
Starting Diffusion Policy training...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch 0 | Loss: 0.0037
Epoch 1 | Loss: 0.0088
Epoch 2 | Loss: 0.0004
Epoch 3 | Loss: 0.0003
Epoch 4 | Loss: 0.0035
Epoch 5 | Loss: 0.0010
